# Dataset exploration
This notebook runs the project pipeline steps in order:
1. fetch raw data
2. extract structure
3. compute features
4. build dataset. 

It shows mocked process of dataset extraction.

In [1]:
import json
from pathlib import Path

import pandas as pd

from src.config import Settings
from src.data import (
    build_dataset,
    compute_features,
    enrich_content,
    extract_structure,
    fetch_raw,
)
from src.data.utils import load_json

In [2]:
nb_cfg = Settings(
    max_repos=25,
    min_stars=500,
)
raw_data_path = Path("data/raw/nb_test.json")
structure_path = Path("data/interim/nb_test.json")
structure_enriched_path = Path("data/interim/nb_test_enriched.json")
computed_features_path = Path("data/interim/nb_test_features.json")
dataset_output_dir = Path("data/processed/nb_test")

In [3]:
def show_item_from_json(file_path: Path, index: int = 1):
    """Utility function to display a specific item from a JSON file."""
    with open(file_path) as f:
        data = json.load(f)

    if isinstance(data, list) and len(data) > index:
        entry = data[index]

    elif isinstance(data, dict):
        entry = next(iter(data.values())) if data else None
    else:
        entry = None

    if entry:
        print(json.dumps(entry, indent=2))
    else:
        print(f"No data found in {file_path}")

In [4]:
fetch_raw(cfg=nb_cfg, output_path=raw_data_path)

2026-04-09 16:53:24,301 [INFO] base_query: 'language:Python size:10..500000 pushed:>=2025-04-09 fork:false archived:false'  min_stars: 500  target: 25
2026-04-09 16:53:24,302 [INFO] Starting greedy batching to collect 25 repos (min_stars=500)
2026-04-09 16:53:24,302 [INFO] Batch 1: Fetching repos with stars:>=500 (remaining: 25)
2026-04-09 16:53:24,304 [DEBUG] Starting new HTTPS connection (1): api.github.com:443
2026-04-09 16:53:27,758 [DEBUG] https://api.github.com:443 "GET /search/repositories?q=language%3APython+size%3A10..500000+pushed%3A%3E%3D2025-04-09+fork%3Afalse+archived%3Afalse+stars%3A%3E%3D500&sort=stars&order=desc&per_page=100&page=1 HTTP/1.1" 200 None
2026-04-09 16:53:27,839 [DEBUG]   page 1: 100 results (batch total: 100)
2026-04-09 16:53:27,839 [INFO] Batch 1 complete: +25 repos (total: 25/25)
2026-04-09 16:53:27,840 [INFO] Greedy batching complete: 25 repos collected in 2 batch(es)
2026-04-09 16:53:27,841 [INFO] collected 25 repos
2026-04-09 16:53:27,841 [INFO] Shuffl

In [5]:
show_item_from_json(file_path=raw_data_path, index=1)

{
  "repo": {
    "id": 589831718,
    "node_id": "R_kgDOIygeJg",
    "name": "ComfyUI",
    "full_name": "Comfy-Org/ComfyUI",
    "private": false,
    "owner": {
      "login": "Comfy-Org",
      "id": 166579949,
      "node_id": "O_kgDOCe3O7Q",
      "avatar_url": "https://avatars.githubusercontent.com/u/166579949?v=4",
      "gravatar_id": "",
      "url": "https://api.github.com/users/Comfy-Org",
      "html_url": "https://github.com/Comfy-Org",
      "followers_url": "https://api.github.com/users/Comfy-Org/followers",
      "following_url": "https://api.github.com/users/Comfy-Org/following{/other_user}",
      "gists_url": "https://api.github.com/users/Comfy-Org/gists{/gist_id}",
      "starred_url": "https://api.github.com/users/Comfy-Org/starred{/owner}{/repo}",
      "subscriptions_url": "https://api.github.com/users/Comfy-Org/subscriptions",
      "organizations_url": "https://api.github.com/users/Comfy-Org/orgs",
      "repos_url": "https://api.github.com/users/Comfy-Org/rep

In [6]:
extract_structure(input_path=raw_data_path, output_path=structure_path)

2026-04-09 16:54:17,906 [DEBUG] Loaded 25 records from data/raw/nb_test.json
2026-04-09 16:54:17,907 [INFO] Processing 25 raw records...
2026-04-09 16:54:17,913 [INFO] [1/25] anthropics/skills: OK
2026-04-09 16:54:17,920 [INFO] [2/25] Comfy-Org/ComfyUI: OK
2026-04-09 16:54:17,930 [INFO] [3/25] ytdl-org/youtube-dl: OK
2026-04-09 16:54:17,935 [INFO] [4/25] browser-use/browser-use: OK
2026-04-09 16:54:17,937 [INFO] [5/25] vinta/awesome-python: OK
2026-04-09 16:54:17,941 [INFO] [6/25] bregman-arie/devops-exercises: OK
2026-04-09 16:54:17,965 [INFO] [7/25] fastapi/fastapi: OK
2026-04-09 16:54:17,966 [INFO] [8/25] public-apis/public-apis: OK
2026-04-09 16:54:17,968 [INFO] [9/25] 3b1b/manim: OK
2026-04-09 16:54:17,978 [INFO] [10/25] yt-dlp/yt-dlp: OK
2026-04-09 16:54:17,980 [INFO] [11/25] EbookFoundation/free-programming-books: OK
2026-04-09 16:54:18,013 [INFO] [12/25] Significant-Gravitas/AutoGPT: OK
2026-04-09 16:54:18,014 [INFO] [13/25] openai/whisper: OK
2026-04-09 16:54:18,064 [INFO] [14

In [7]:
show_item_from_json(file_path=structure_path, index=0)

{
  "repo_url": "https://github.com/anthropics/skills",
  "stars": 113814,
  "forks": 12966,
  "created_at": "2025-09-22T15:53:31Z",
  "last_updated": "2026-04-09T16:46:17Z",
  "repo_size_kb": 3470,
  "open_issues_count": 669,
  "language": "Python",
  "tree_truncated": false,
  "py_files": [
    {
      "path": "skills/docx/scripts/__init__.py",
      "size": 1
    },
    {
      "path": "skills/docx/scripts/accept_changes.py",
      "size": 4051
    },
    {
      "path": "skills/docx/scripts/comment.py",
      "size": 10694
    },
    {
      "path": "skills/docx/scripts/office/helpers/__init__.py",
      "size": 0
    },
    {
      "path": "skills/docx/scripts/office/helpers/merge_runs.py",
      "size": 5567
    },
    {
      "path": "skills/docx/scripts/office/helpers/simplify_redlines.py",
      "size": 5754
    },
    {
      "path": "skills/docx/scripts/office/pack.py",
      "size": 4991
    },
    {
      "path": "skills/docx/scripts/office/soffice.py",
      "size": 5301


In [8]:
enrich_content(input_path=structure_path, output_path=structure_enriched_path)

2026-04-09 16:54:18,371 [DEBUG] Loaded 25 records from data/interim/nb_test.json
2026-04-09 16:54:18,372 [INFO] Enriching 25 records...


2026-04-09 16:54:18,842 [DEBUG] https://api.github.com:443 "GET /repos/anthropics/skills/tarball/HEAD HTTP/1.1" 302 0
2026-04-09 16:54:18,844 [DEBUG] Starting new HTTPS connection (1): codeload.github.com:443
2026-04-09 16:54:19,142 [DEBUG] https://codeload.github.com:443 "GET /anthropics/skills/legacy.tar.gz/HEAD HTTP/1.1" 200 3517336
2026-04-09 16:54:19,706 [INFO] [1/25] https://github.com/anthropics/skills: OK
2026-04-09 16:54:19,995 [DEBUG] https://api.github.com:443 "GET /repos/Comfy-Org/ComfyUI/tarball/HEAD HTTP/1.1" 302 0
2026-04-09 16:54:20,225 [DEBUG] https://codeload.github.com:443 "GET /Comfy-Org/ComfyUI/legacy.tar.gz/HEAD HTTP/1.1" 200 None
2026-04-09 16:54:22,294 [INFO] [2/25] https://github.com/Comfy-Org/ComfyUI: OK
2026-04-09 16:54:22,585 [DEBUG] https://api.github.com:443 "GET /repos/ytdl-org/youtube-dl/tarball/HEAD HTTP/1.1" 302 0
2026-04-09 16:54:22,832 [DEBUG] https://codeload.github.com:443 "GET /ytdl-org/youtube-dl/legacy.tar.gz/HEAD HTTP/1.1" 200 None
2026-04-09 1

In [9]:
show_item_from_json(file_path=structure_enriched_path, index=0)

{
  "repo_url": "https://github.com/anthropics/skills",
  "stars": 113814,
  "forks": 12966,
  "created_at": "2025-09-22T15:53:31Z",
  "last_updated": "2026-04-09T16:46:17Z",
  "repo_size_kb": 3470,
  "open_issues_count": 669,
  "language": "Python",
  "tree_truncated": false,
  "py_files": [
    {
      "path": "skills/docx/scripts/__init__.py",
      "size": 1
    },
    {
      "path": "skills/docx/scripts/accept_changes.py",
      "size": 4051
    },
    {
      "path": "skills/docx/scripts/comment.py",
      "size": 10694
    },
    {
      "path": "skills/docx/scripts/office/helpers/__init__.py",
      "size": 0
    },
    {
      "path": "skills/docx/scripts/office/helpers/merge_runs.py",
      "size": 5567
    },
    {
      "path": "skills/docx/scripts/office/helpers/simplify_redlines.py",
      "size": 5754
    },
    {
      "path": "skills/docx/scripts/office/pack.py",
      "size": 4991
    },
    {
      "path": "skills/docx/scripts/office/soffice.py",
      "size": 5301


In [10]:
compute_features(input_path=structure_enriched_path, output_path=computed_features_path)

2026-04-09 16:55:17,652 [DEBUG] Loaded 25 records from data/interim/nb_test_enriched.json
2026-04-09 16:55:17,653 [INFO] Computing features for 25 records...
2026-04-09 16:55:17,653 [INFO] [1/25] https://github.com/anthropics/skills: OK
2026-04-09 16:55:17,654 [INFO] [2/25] https://github.com/Comfy-Org/ComfyUI: OK
2026-04-09 16:55:17,654 [INFO] [3/25] https://github.com/ytdl-org/youtube-dl: OK
2026-04-09 16:55:17,655 [INFO] [4/25] https://github.com/browser-use/browser-use: OK
2026-04-09 16:55:17,656 [INFO] [5/25] https://github.com/vinta/awesome-python: OK
2026-04-09 16:55:17,656 [INFO] [6/25] https://github.com/bregman-arie/devops-exercises: OK
2026-04-09 16:55:17,657 [INFO] [7/25] https://github.com/fastapi/fastapi: OK
2026-04-09 16:55:17,657 [INFO] [8/25] https://github.com/public-apis/public-apis: OK
2026-04-09 16:55:17,658 [INFO] [9/25] https://github.com/3b1b/manim: OK
2026-04-09 16:55:17,658 [INFO] [10/25] https://github.com/yt-dlp/yt-dlp: OK
2026-04-09 16:55:17,659 [INFO] [11/

In [11]:
show_item_from_json(file_path=computed_features_path, index=0)

{
  "repo_url": "https://github.com/anthropics/skills",
  "stars": 113814,
  "forks": 12966,
  "repo_size_kb": 3470,
  "open_issues_count": 669,
  "num_files": 388,
  "num_py_files": 72,
  "num_js_files": 1,
  "num_ts_files": 0,
  "num_html_files": 3,
  "num_css_files": 0,
  "num_json_files": 1,
  "num_sh_files": 2,
  "num_yaml_files": 0,
  "num_test_files": 0,
  "test_file_ratio": 0.0,
  "num_docs_files": 84,
  "num_notebooks": 0,
  "other_extensions_count": 225,
  "has_dedicated_test_dir": false,
  "has_license": false,
  "has_src_dir": false,
  "has_docs_dir": false,
  "has_scripts_dir": false,
  "num_dirs": 85,
  "avg_files_per_dir": 4.51,
  "avg_py_file_len": 199.3,
  "avg_test_file_len": 0,
  "avg_nb_cell_count": 0,
  "avg_docs_file_len": 241.3,
  "num_dependencies": 6,
  "repo_age_days": 199,
  "recent_activity_days": 0,
  "has_pyproject_toml": false,
  "has_dockerfile": false,
  "has_github_actions": false,
  "has_requirements_txt": false,
  "has_conda_env_file": false,
  "has_

In [12]:
build_dataset(input_path=computed_features_path, output_dir=dataset_output_dir)

2026-04-09 16:55:17,693 [INFO] Loading features from data/interim/nb_test_features.json...
2026-04-09 16:55:17,693 [DEBUG] Loaded 25 records from data/interim/nb_test_features.json
2026-04-09 16:55:17,694 [INFO] Loaded 25 records
2026-04-09 16:55:17,695 [INFO] Label distribution in dataset:
2026-04-09 16:55:17,696 [INFO]   has_pyproject_toml: 16/25 (64.0%)
2026-04-09 16:55:17,697 [INFO]   has_dockerfile: 8/25 (32.0%)
2026-04-09 16:55:17,698 [INFO]   has_github_actions: 21/25 (84.0%)
2026-04-09 16:55:17,702 [INFO] Saved dataset to data/processed/nb_test/v1.0.4/dataset.csv
2026-04-09 16:55:17,702 [INFO] Dataset checksum: sha256:3749ca56ab041665aa729b36ead78719576aad771caac83407480e2c54d50421
2026-04-09 16:55:17,705 [INFO] Saved 1 records -> data/processed/nb_test/v1.0.4/manifest.json
2026-04-09 16:55:17,705 [INFO] Saved manifest to data/processed/nb_test/v1.0.4/manifest.json
2026-04-09 16:55:17,706 [INFO] ============================================================
2026-04-09 16:55:17,70

In [13]:
versions = sorted([d for d in dataset_output_dir.iterdir() if d.is_dir()], reverse=True)
latest_version = versions[0]
dataset_path = latest_version / "dataset.csv"
manifest_path = latest_version / "manifest.json"

In [14]:
df = pd.read_csv(dataset_path)
print("Dataset preview:")
df.head()

Dataset preview:


,repo_url,stars,forks,repo_size_kb,open_issues_count,num_files,num_py_files,num_js_files,num_ts_files,num_html_files,...,has_pyproject_toml,has_dockerfile,has_github_actions,has_requirements_txt,has_conda_env_file,has_docker_compose,has_precommit_config,has_setup_py,has_tox_ini,has_makefile
0,https://github.com/3b1b/manim,85910,7204,76630,482,167,95,0,0,0,...,True,False,True,True,False,False,False,True,False,False
1,https://github.com/521xueweihan/HelloGitHub,149828,11382,9730,318,254,2,0,0,0,...,False,False,False,False,False,False,False,False,False,False
2,https://github.com/AUTOMATIC1111/stable-diffus...,162238,30245,36552,2478,315,207,30,0,11,...,True,False,True,True,True,False,False,False,False,False
3,https://github.com/Comfy-Org/ComfyUI,108234,12530,86749,3947,776,516,0,0,0,...,True,False,True,True,False,False,False,False,False,False
4,https://github.com/EbookFoundation/free-progra...,385242,66097,21200,79,233,1,0,0,1,...,False,False,True,False,False,False,False,False,False,False


In [15]:
manifest = load_json(manifest_path)
print("Manifest content:")
print(manifest)

Manifest content:
{'version': '1.0.4', 'created_at': '2026-04-09T16:55:17.702987+00:00', 'script': 'src/data/build_dataset.py', 'git_commit': 'b79419dff250cc3a7a861447e55c9519b0c22d50', 'input_source': 'data/interim/nb_test_features.json', 'pipeline_stats': {'extraction_log': 'extraction_20260409_165324.log'}, 'schema': {'total_features': 43, 'total_rows': 25, 'feature_columns': ['repo_url', 'stars', 'forks', 'repo_size_kb', 'open_issues_count', 'num_files', 'num_py_files', 'num_js_files', 'num_ts_files', 'num_html_files', 'num_css_files', 'num_json_files', 'num_sh_files', 'num_yaml_files', 'num_test_files', 'test_file_ratio', 'num_docs_files', 'num_notebooks', 'other_extensions_count', 'has_dedicated_test_dir', 'has_license', 'has_src_dir', 'has_docs_dir', 'has_scripts_dir', 'num_dirs', 'avg_files_per_dir', 'avg_py_file_len', 'avg_test_file_len', 'avg_nb_cell_count', 'avg_docs_file_len', 'num_dependencies', 'repo_age_days', 'recent_activity_days', 'has_pyproject_toml', 'has_dockerfile